# Fretwork — Alternate Tunings, Capo & Preferred Playing Position

Feature layer on top of the fret-assignment pipeline. Adds three capabilities through a
**single insertion point** — the fretboard that `get_possible_positions` reads from — so every
existing `assign_*` method becomes tuning/capo/position aware with no edits to those methods.

1. **Alternate tunings** (Drop D, Drop C, DADGAD, …) and **capo** — one parameterization: an
   open-string pitch vector plus a capo-relative fret window.
2. **Preferred playing position** — a soft region anchor (a *preference*, not a penalty): pulls
   each onset group's hand-center toward a target fret, with a free window, so a clean stepwise
   reason to leave the region can still win.
3. **End-to-end demo** + **correctness tests** that need no ground truth.

**Scope honesty.** Capo is exactly correct (pure transposition + fret window; all costs are
position-relative). Alternate tunings are correct for candidate generation and any geometry-based
assigner; off standard tuning the *learned* position prior goes inert (its keys are standard-tuning
triples), so geometry — and the position anchor below — govern register. These ship as a tested
**capability**, not a reported accuracy number, since there is no Drop-D/capo ground truth.

## 1. Setup — import the base pipeline

`%run` the base notebook to bring in the fretboard, candidate generation, every `assign_*` method,
`enrich_notes_with_context`, the `records` / TRAIN-VAL-TEST splits, the eval harness, and the
position prior.

**One path, one place.** Edit `BASE_NB` below — nothing else. The run cell uses that same variable
(via `run_line_magic`, so there's no `%run "$VAR"` → appends-`.py` problem). It mounts Drive if
needed, and if the file isn't found it prints the folder's real contents — so a wrong name,
extension, or a shared-in shortcut is obvious.

In [1]:
from pathlib import Path

# Mount Drive if needed (idempotent — safe to re-run).
if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

# === THE ONLY PATH YOU EDIT ===
BASE_NB = "/content/drive/MyDrive/Capstone/Code/fret_algo_combined_tuned_heldout_LATEST.ipynb"

p = Path(BASE_NB)
if not p.exists():
    print("NOT FOUND:\n  ", BASE_NB, "\n")
    if p.parent.exists():
        print(f"Actual contents of {p.parent}  (repr reveals hidden trailing spaces):")
        for item in sorted(p.parent.iterdir()):
            print("   ", repr(item.name))
    else:
        cur = p.parent
        while not cur.exists() and cur != cur.parent:
            cur = cur.parent
        print(f"Folder {p.parent} does not exist. Deepest existing ancestor: {cur}")
        if cur.exists():
            print("   contains:", [x.name for x in sorted(cur.iterdir())][:40])
    raise FileNotFoundError("Edit BASE_NB to match a name printed above.")

print("Found base notebook:", BASE_NB)

Mounted at /content/drive
Found base notebook: /content/drive/MyDrive/Capstone/Code/fret_algo_combined_tuned_heldout_LATEST.ipynb


In [2]:
# Runs the base end-to-end (including its full eval loop) using the SAME BASE_NB variable —
# no second path to keep in sync. run_line_magic passes the literal string, so the
# "%run $VAR appends .py" gotcha doesn't apply.
get_ipython().run_line_magic('run', f'"{BASE_NB}"')

# Sanity check that the pieces we build on are now in scope.
for _name in ["build_fretboard", "get_possible_positions", "candidate_groups_for_notes",
              "assign_viterbi_playability", "enrich_notes_with_context", "records",
              "estimate_hand_position_from_frets", "OPEN_STRING_MIDI", "MAX_FRET"]:
    assert _name in globals(), f"Expected '{_name}' from the base notebook — did the base run fully?"
print("Base pipeline loaded. Records available:", len(records))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.
OUTPUT_DIR: /content/drive/MyDrive/Capstone/outputs/fretboard_playability
OUTPUT_DIR exists: True

Data root candidates visible to this runtime:
 - /content/drive/MyDrive/Capstone/FullGuitarSetData | exists: True
 - /content/drive/MyDrive/FullGuitarSetData | exists: False
 - /content/drive/MyDrive/Capstone/GuitarSet | exists: True
 - /content/drive/MyDrive/GuitarSet | exists: False
 - /content/drive/MyDrive/Capstone | exists: True
 - /content/drive/MyDrive | exists: True
 - /mnt/data/fretwork_repo/GuitarSet | exists: False
 - /mnt/data/fretwork_repo | exists: False

Top-level MyDrive folders/files visible to Colab:
 - labels.csv
 - yelp_dataset.tar
 - Spring 2025
 - Tell a compelling story. Example. Fall 2020. Airline Pricing.gdoc
 - DATASCI200 Project Proposal.gdoc
 - personalized bus routes.fall 2024.gdoc
 - Final Report T

,string,string_name,fret,midi,pitch_class
0,0,low_E,0,40,4
1,0,low_E,1,41,5
2,0,low_E,2,42,6
3,0,low_E,3,43,7
4,0,low_E,4,44,8
5,0,low_E,5,45,9
6,0,low_E,6,46,10
7,0,low_E,7,47,11
8,0,low_E,8,48,0
9,0,low_E,9,49,1


Example positions for MIDI 64 / E4:


,string,string_name,fret,midi,pitch_class
0,0,low_E,24,64,4
1,1,A,19,64,4
2,2,D,14,64,4
3,3,G,9,64,4
4,4,B,5,64,4
5,5,high_E,0,64,4


,key,root,root_pc,mode,scale_pcs,scale_notes,diatonic_chords
0,C major,C,0,major,"[0, 2, 4, 5, 7, 9, 11]","[C, D, E, F, G, A, B]","[{'degree': 1, 'root_pc': 0, 'root': 'C', 'qua..."
1,C minor,C,0,minor,"[0, 2, 3, 5, 7, 8, 10]","[C, D, D#, F, G, G#, A#]","[{'degree': 1, 'root_pc': 0, 'root': 'C', 'qua..."
2,C# major,C#,1,major,"[1, 3, 5, 6, 8, 10, 0]","[C#, D#, F, F#, G#, A#, C]","[{'degree': 1, 'root_pc': 1, 'root': 'C#', 'qu..."
3,C# minor,C#,1,minor,"[1, 3, 4, 6, 8, 9, 11]","[C#, D#, E, F#, G#, A, B]","[{'degree': 1, 'root_pc': 1, 'root': 'C#', 'qu..."
4,D major,D,2,major,"[2, 4, 6, 7, 9, 11, 1]","[D, E, F#, G, A, B, C#]","[{'degree': 1, 'root_pc': 2, 'root': 'D', 'qua..."


...

Saved compact report table to: /content/drive/MyDrive/Capstone/outputs/fretboard_playability/fretboard_report_table_heldout_test.csv


,n_notes,exact_position_acc,string_acc,fret_acc,avg_fret_error,avg_string_error,correct_pitch_from_tab_rate,valid_position_rate,duplicate_string_violation_rate,avg_group_span,avg_fret_jump,avg_string_jump,large_jump_rate,large_jump_count,recording,method,is_solo,is_comp,eval_set
0,58,0.620690,0.620690,0.620690,2.155172,0.448276,1.0,1.0,0.0,0.087719,1.357143,0.482143,0.000000,0,00_Funk1-114-Ab_solo,combined_all,True,False,heldout_test
1,58,0.603448,0.603448,0.603448,2.293103,0.465517,1.0,1.0,0.0,0.087719,1.357143,0.535714,0.000000,0,00_Funk1-114-Ab_solo,combined_all_tuned,True,False,heldout_test
2,58,0.241379,0.241379,0.241379,4.120690,0.844828,1.0,1.0,0.0,0.000000,1.553571,0.660714,0.000000,0,00_Funk1-114-Ab_solo,highest_string,True,False,heldout_test
3,58,0.241379,0.241379,0.241379,4.120690,0.844828,1.0,1.0,0.0,0.000000,1.553571,0.660714,0.000000,0,00_Funk1-114-Ab_solo,lowest_fret,True,False,heldout_test
4,58,0.517241,0.517241,0.517241,2.724138,0.551724,1.0,1.0,0.0,0.087719,1.446429,0.553571,0.017857,1,00_Funk1-114-Ab_solo,nearest_previous,True,False,heldout_test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427,93,0.215054,0.215054,0.215054,4.365591,0.946237,1.0,1.0,0.0,0.000000,1.717391,0.293478,0.000000,0,05_SS2-88-F_solo,lowest_fret,True,False,heldout_test
428,93,0.182796,0.182796,0.182796,7.215054,1.526882,1.0,1.0,0.0,0.000000,1.391304,0.358696,0.000000,0,05_SS2-88-F_solo,nearest_previous,True,False,heldout_test
429,93,0.494624,0.494624,0.494624,2.462366,0.537634,1.0,1.0,0.0,0.000000,1.695652,0.467391,0.032609,3,05_SS2-88-F_solo,old_music_theory_greedy,True,False,heldout_test
430,93,0.430108,0.430108,0.430108,3.043011,0.655914,1.0,1.0,0.0,0.000000,1.315217,0.369565,0.000000,0,05_SS2-88-F_solo,viterbi_original,True,False,heldout_test


Saved error examples: /content/drive/MyDrive/Capstone/outputs/fretboard_playability/fretboard_error_examples.csv


,recording,start,midi,chord_label,key_label,true_string,true_fret,pred_string,pred_fret,fret_error,string_error,correct_pitch_from_tab
74468,00_Funk1-114-Ab_solo,1.109204,47,G#:maj,Ab:major,0,7,1,2,5,1,True
74469,00_Funk1-114-Ab_solo,1.335122,48,G#:maj,Ab:major,0,8,1,3,5,1,True
74471,00_Funk1-114-Ab_solo,1.587117,56,G#:maj,Ab:major,2,6,3,1,5,1,True
74486,00_Funk1-114-Ab_solo,7.105644,56,G#:maj,Ab:major,2,6,1,11,5,1,True
74487,00_Funk1-114-Ab_solo,7.380519,58,G#:maj,Ab:major,2,8,1,13,5,1,True
74488,00_Funk1-114-Ab_solo,7.644396,59,G#:maj,Ab:major,2,9,1,14,5,1,True
74489,00_Funk1-114-Ab_solo,7.918138,58,G#:maj,Ab:major,2,8,1,13,5,1,True
74490,00_Funk1-114-Ab_solo,8.168546,56,G#:maj,Ab:major,2,6,1,11,5,1,True
74491,00_Funk1-114-Ab_solo,8.417775,56,G#:maj,Ab:major,2,6,1,11,5,1,True
74492,00_Funk1-114-Ab_solo,8.504759,58,C#:maj,Ab:major,2,8,1,13,5,1,True


Final output folder:
/content/drive/MyDrive/Capstone/outputs/fretboard_playability

CSV files currently in output folder:
 - fretboard_error_examples.csv
 - fretboard_eval_by_recording.csv
 - fretboard_eval_by_recording_heldout_test.csv
 - fretboard_eval_summary_all.csv
 - fretboard_eval_summary_heldout_test.csv
 - fretboard_eval_summary_solo.csv
 - fretboard_eval_summary_solo_heldout_test.csv
 - fretboard_failed_runs.csv
 - fretboard_failed_runs_heldout_test.csv
 - fretboard_method_comparison_table.csv
 - fretboard_method_comparison_table_heldout_test.csv
 - fretboard_method_comparison_table_solo.csv
 - fretboard_method_comparison_table_solo_heldout_test.csv
 - fretboard_predictions_all.csv
 - fretboard_predictions_heldout_test.csv
 - fretboard_report_table_heldout_test.csv
 - fretboard_train_val_test_split.csv
 - fretboard_tuning_results_validation.csv
Base pipeline loaded. Records available: 360


## 2. Alternate tunings + capo

One parameterization. A tuning is six open-string MIDI values (index 0 = low string … 5 = high
string). A capo on fret *N* raises every open string by *N* **and** becomes the new fret 0, so
frets are reported capo-relative (displayed fret `f` = physical fret `f + N`) — the way guitarists
read capo tab.

`apply_fretboard(...)` rebuilds the global fretboard the rest of the pipeline reads from. Because
candidate generation routes through `get_possible_positions`, this is the only thing that has to
change. Call it **after** the base has built its position prior (it has, during `%run`); the prior
is standard-tuning specific and must not be rebuilt per tuning.

In [3]:
from collections import defaultdict
from itertools import product
import math
import numpy as np
import pandas as pd

STANDARD_TUNING = [40, 45, 50, 55, 59, 64]   # E2 A2 D3 G3 B3 E4 — reference, do not mutate

TUNINGS = {
    'standard':    [40, 45, 50, 55, 59, 64],  # E A D G B E
    'drop_d':      [38, 45, 50, 55, 59, 64],  # D A D G B E
    'drop_c':      [36, 43, 48, 53, 57, 62],  # C G C F A D
    'eb_standard': [39, 44, 49, 54, 58, 63],  # half-step down
    'dadgad':      [38, 45, 50, 55, 57, 62],  # D A D G A D
    'open_g':      [38, 43, 50, 55, 59, 62],  # D G D G B D
}

ACTIVE_TUNING_NAME = 'standard'
ACTIVE_OPEN_MIDI   = list(STANDARD_TUNING)   # capo-shifted effective open pitches
CAPO               = 0
MAX_FRET_DISPLAYED = MAX_FRET

def _midi_name(m):
    names = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
    return f'{names[m % 12]}{m // 12 - 1}'

def tab_capo_header():
    parts = []
    if ACTIVE_TUNING_NAME != 'standard':
        parts.append(f'Tuning: {ACTIVE_TUNING_NAME}')
    if CAPO:
        parts.append(f'Capo {CAPO}')
    return ' | '.join(parts) if parts else 'Standard tuning, no capo'

def apply_fretboard(tuning='standard', capo=0, max_fret=MAX_FRET, verbose=True):
    """Rebuild the global fretboard for a given tuning + capo.

    tuning : key in TUNINGS, or an explicit 6-element list of open-string MIDI values.
    capo   : fret the capo sits on. Frets are reported RELATIVE TO THE CAPO (capo = fret 0);
             displayed fret f maps to physical fret f + capo.

    Rebinds OPEN_STRING_MIDI, MIDI_TO_POSITIONS, fretboard_df, MAX_FRET_DISPLAYED, CAPO, ACTIVE_*.
    Call AFTER the prior is built; do not rebuild the prior after switching tunings.
    """
    global OPEN_STRING_MIDI, MIDI_TO_POSITIONS, fretboard_df
    global ACTIVE_TUNING_NAME, ACTIVE_OPEN_MIDI, CAPO, MAX_FRET_DISPLAYED

    base = TUNINGS[tuning] if isinstance(tuning, str) else list(tuning)
    assert len(base) == 6, 'tuning must have 6 open-string MIDI values'
    assert 0 <= capo <= max_fret - 1, 'capo out of range'

    open_eff = [m + capo for m in base]
    max_fret_displayed = max_fret - capo

    fb = build_fretboard(open_string_midi=open_eff, max_fret=max_fret_displayed)
    m2p = defaultdict(list)
    for row in fb.to_dict('records'):
        m2p[int(row['midi'])].append({
            'string': int(row['string']), 'string_name': row['string_name'],
            'fret': int(row['fret']), 'midi': int(row['midi']),
            'pitch_class': int(row['pitch_class']),
        })

    OPEN_STRING_MIDI   = open_eff
    MIDI_TO_POSITIONS  = m2p
    fretboard_df       = fb
    MAX_FRET_DISPLAYED = max_fret_displayed
    CAPO               = capo
    ACTIVE_TUNING_NAME = tuning if isinstance(tuning, str) else 'custom'
    ACTIVE_OPEN_MIDI   = open_eff

    if verbose:
        names = ' '.join(_midi_name(m) for m in base)
        print(f'Fretboard set: tuning={ACTIVE_TUNING_NAME} [{names}], capo={capo}')
        print(f'  effective open MIDI = {open_eff}')
        print(f'  displayed frets 0..{max_fret_displayed} (physical {capo}..{max_fret})')
        print(f'  tab header => {tab_capo_header()}')

apply_fretboard('standard', capo=0)   # establish defaults; identical to original behavior

Fretboard set: tuning=standard [E2 A2 D3 G3 B3 E4], capo=0
  effective open MIDI = [40, 45, 50, 55, 59, 64]
  displayed frets 0..24 (physical 0..24)
  tab header => Standard tuning, no capo


## 3. Preferred playing position

A *preference*, not a penalty. The cost is on **deviation from a target region**, zero inside a
free window, so a hand sitting at the target pays nothing and a hand with a genuine reason to be
elsewhere (a smooth stepwise run) can still win. It lives in **base cost**, orthogonal to the
transition/movement costs (which the error analysis already showed are saturated).

`PREF_POSITION_WEIGHT = 0` leaves it off — in standard tuning the learned prior governs register.
Turn it on for user control ("play around the 9th fret"), or off-standard where it's the only
register anchor.

In [4]:
PREF_POSITION_TARGET = 0
PREF_POSITION_WEIGHT = 0.0      # 0 = OFF
PREF_POSITION_WINDOW = 2        # frets of free reach each side of the target

POSITION_PRESETS = {
    'open': 2, 'fret_5': 5, 'fret_7': 7, 'fret_9': 9, 'fret_12': 12, 'high': 14,
}

def set_preferred_position(where=None, weight=0.30, window=2, verbose=True):
    """where: preset name, an integer target fret, or None/'none' to switch the feature off."""
    global PREF_POSITION_TARGET, PREF_POSITION_WEIGHT, PREF_POSITION_WINDOW
    if where is None or where == 'none':
        PREF_POSITION_WEIGHT = 0.0
        if verbose: print('Preferred position: OFF (learned prior governs register)')
        return
    target = POSITION_PRESETS[where] if isinstance(where, str) else int(where)
    PREF_POSITION_TARGET, PREF_POSITION_WEIGHT, PREF_POSITION_WINDOW = target, float(weight), int(window)
    if verbose:
        lo, hi = max(0, target - window), target + window
        print(f'Preferred position: target fret {target}, free window {lo}..{hi}, weight {weight}')

def preferred_position_cost(group_positions):
    """Deadband-linear pull toward PREF_POSITION_TARGET. Zero inside the window, zero for
    all-open groups (no hand position to anchor), zero when weight is 0."""
    if PREF_POSITION_WEIGHT <= 0 or not group_positions:
        return 0.0
    frets = [p['fret'] for p in group_positions]
    if not any(f > 0 for f in frets):
        return 0.0
    center = estimate_hand_position_from_frets(frets)
    dist = abs(center - PREF_POSITION_TARGET)
    return 0.0 if dist <= PREF_POSITION_WINDOW else PREF_POSITION_WEIGHT * (dist - PREF_POSITION_WINDOW)

## 4. Wire the position preference into candidate scoring

Redefine `candidate_groups_for_notes` (overriding the base copy) to add the position term to
`base_cost` in both scoring spots, **before** the top-`max_candidates` truncation so a
position-favorable candidate can't be cut first. Everything else is identical to the base.
Every assigner that routes through this function now respects the anchor.

In [5]:
def candidate_groups_for_notes(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos:
            return []
        position_lists.append(pos)
    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
            continue
        base_cost = (group_playability_cost(combo)
                     + context_cost(group_notes, combo)
                     + preferred_position_cost(combo))          # <-- position preference
        if math.isfinite(base_cost):
            candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))
    if not candidates:
        for combo in product(*position_lists):
            combo = list(combo)
            base_cost = group_playability_cost(combo) + preferred_position_cost(combo)  # <-- here too
            if math.isinf(base_cost):
                base_cost = 1000.0
            candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))
    return sorted(candidates, key=lambda c: c['base_cost'])[:max_candidates]

print('candidate_groups_for_notes overridden with position preference (currently '
      + ('ON' if PREF_POSITION_WEIGHT > 0 else 'OFF') + ').')

candidate_groups_for_notes overridden with position preference (currently OFF).


## 5. Correctness tests (no ground truth)

Properties that must hold regardless of the "correct" fingering: a roundtrip invariant, the capo
transpose-equivalence (the headline evidence capo is correct), a Drop-D functional probe, and capo
masking. Run on `assign_viterbi_playability` — geometry only, no absolute-pitch prior — so the
equivalences hold exactly. The position anchor is forced OFF here so it can't perturb equivalence.

In [6]:
def _note(midi, start, dur=0.25):
    return {'midi': int(midi), 'start': float(start), 'duration': dur,
            'pitch_class': int(midi) % 12, 'true_string': None, 'true_fret': None}

def _seq(midis, dt=0.5):
    return [_note(m, i * dt) for i, m in enumerate(midis)]

def _frets(notes, assigner):
    rows = sorted(assigner(notes), key=lambda r: (r['start'], r['midi']))
    return [(r['pred_string'], r['pred_fret']) for r in rows]

MELODY = [64, 67, 69, 71, 72, 71, 69, 67, 64]   # mid-neck line, reachable in every config below

def test_roundtrip_pitch_invariant(assigner=assign_viterbi_playability):
    for r in assigner(_seq(MELODY)):
        s, f = r['pred_string'], r['pred_fret']
        assert s is not None and f is not None, f'unassigned note {r["midi"]}'
        assert ACTIVE_OPEN_MIDI[s] + f == r['midi'], f'pitch mismatch on {r["midi"]}'
        assert 0 <= f <= MAX_FRET_DISPLAYED, f'fret {f} outside capo window'
    return True

def test_capo_transpose_equivalence(capo=2, assigner=assign_viterbi_playability):
    apply_fretboard('standard', 0, verbose=False)
    base = _frets(_seq(MELODY), assigner)
    apply_fretboard('standard', capo, verbose=False)
    shifted = _frets(_seq([m + capo for m in MELODY]), assigner)
    apply_fretboard('standard', 0, verbose=False)
    assert base == shifted, f'capo {capo} not transpose-equivalent:\n{base}\n{shifted}'
    return True

def test_drop_d_low_note():
    apply_fretboard('standard', 0, verbose=False)
    assert get_possible_positions(38) == [], 'D2 should be unreachable in standard'
    apply_fretboard('drop_d', 0, verbose=False)
    pos = get_possible_positions(38)
    apply_fretboard('standard', 0, verbose=False)
    assert any(p['string'] == 0 and p['fret'] == 0 for p in pos), 'D2 should be open low string in Drop D'
    return True

def test_capo_masks_low_pitches(capo=5):
    apply_fretboard('standard', capo, verbose=False)
    lowest = STANDARD_TUNING[0] + capo
    below, at = get_possible_positions(lowest - 1), get_possible_positions(lowest)
    apply_fretboard('standard', 0, verbose=False)
    assert below == [], f'{lowest - 1} should be unreachable with capo {capo}'
    assert any(p['fret'] == 0 for p in at), 'lowest capoed pitch should be an open string'
    return True

_saved_weight = PREF_POSITION_WEIGHT
set_preferred_position(None, verbose=False)          # anchor OFF during equivalence tests
print('Running tuning/capo tests...\n')
for name, fn in [
    ('roundtrip (standard)',        test_roundtrip_pitch_invariant),
    ('capo transpose equivalence',  test_capo_transpose_equivalence),
    ('drop-D low note functional',  test_drop_d_low_note),
    ('capo masks low pitches',      test_capo_masks_low_pitches),
]:
    try:
        fn(); print(f'  PASS  {name}')
    except AssertionError as e:
        print(f'  FAIL  {name} :: {e}')

for t, c in [('drop_d', 0), ('drop_c', 0), ('standard', 3), ('dadgad', 2)]:
    apply_fretboard(t, c, verbose=False)
    try:
        test_roundtrip_pitch_invariant(); print(f'  PASS  roundtrip ({t}, capo {c})')
    except AssertionError as e:
        print(f'  FAIL  roundtrip ({t}, capo {c}) :: {e}')

apply_fretboard('standard', 0, verbose=False)
PREF_POSITION_WEIGHT = _saved_weight                 # restore whatever the anchor was

Running tuning/capo tests...

  PASS  roundtrip (standard)
  PASS  capo transpose equivalence
  PASS  drop-D low note functional
  PASS  capo masks low pitches
  PASS  roundtrip (drop_d, capo 0)
  PASS  roundtrip (drop_c, capo 0)
  PASS  roundtrip (standard, capo 3)
  PASS  roundtrip (dadgad, capo 2)


## 6. End-to-end demo

Run a real clip through a chosen tuning/capo and render ASCII tab. The demo uses each note's
**pitch + timing** and ignores the standard-tuning ground-truth labels — so it shows how the same
pitches are fingered on a differently tuned/capoed guitar. Uses the base's `render_ascii_tab` if
present, otherwise a compact built-in renderer.

In [7]:
def _reachable_notes(enriched):
    keep, dropped = [], []
    for n in enriched:
        (keep if get_possible_positions(n['midi']) else dropped).append(n)
    return keep, dropped

def _pred_to_parsed(pred_rows, record):
    notes = []
    for r in pred_rows:
        if r.get('pred_string') is None or r.get('pred_fret') is None:
            continue
        notes.append({'string': int(r['pred_string']), 'fret': int(r['pred_fret']),
                      'start': float(r['start']),
                      'duration': float(r.get('duration', 0.5) or 0.5),
                      'midi': int(r['midi'])})
    return {'notes': notes, 'beats': record.get('beats', []), 'tempo': record.get('tempo')}

def _demo_render_tab(parsed, max_notes=None, col_width=3, step=0.25, max_cols=90):
    notes = sorted(parsed['notes'], key=lambda n: (n['start'], n['midi']))
    if not notes:
        return '(no notes)'
    DISPLAY_TO_STRING = [5, 4, 3, 2, 1, 0]; LABELS = ['e', 'B', 'G', 'D', 'A', 'E']
    t0 = notes[0]['start']
    ncols = min(int((notes[-1]['start'] - t0) / step) + 2, max_cols)
    cells = [[None] * ncols for _ in range(6)]
    for n in notes:
        c = min(int(round((n['start'] - t0) / step)), ncols - 1)
        try: row = DISPLAY_TO_STRING.index(n['string'])
        except ValueError: continue
        cells[row][c] = n['fret']
    def fmt(v):
        if v is None: return '-' * col_width
        s = str(v); return s + '-' * (col_width - len(s)) if len(s) < col_width else s[:col_width]
    return '\n'.join(LABELS[r] + '|' + ''.join(fmt(cells[r][c]) for c in range(ncols)) + '|'
                     for r in range(6))

def demo_tab(record, tuning='standard', capo=0,
             assigner=assign_viterbi_playability, max_notes=48):
    apply_fretboard(tuning, capo, verbose=False)
    try:
        enriched = sorted(enrich_notes_with_context(record), key=lambda n: (n['start'], n['midi']))
        if max_notes:
            enriched = enriched[:max_notes]
        playable, dropped = _reachable_notes(enriched)
        if not playable:
            print(f"{record['recording']} | {tab_capo_header()}: no notes playable here."); return None

        used = assigner.__name__
        try:
            pred = assigner(playable)
        except ValueError:
            pred = assign_baseline_lowest_fret(playable)
            used = 'lowest_fret (fallback: a chord had no joint placement)'

        parsed = _pred_to_parsed(pred, record)
        anchor = '' if PREF_POSITION_WEIGHT <= 0 else f'   pref_pos=fret {PREF_POSITION_TARGET} (w={PREF_POSITION_WEIGHT})'
        print('=' * 66)
        print(f"{record['recording']}   |   {tab_capo_header()}")
        line = f"assigner={used}   notes={len(playable)}"
        if dropped: line += f"   dropped_unreachable={len(dropped)}"
        print(line + anchor); print('=' * 66)
        renderer = render_ascii_tab if 'render_ascii_tab' in globals() else _demo_render_tab
        print(renderer(parsed, max_notes=max_notes))
        return parsed
    finally:
        apply_fretboard('standard', 0, verbose=False)

In [ ]:
# Same clip, three configurations. Pick a melodic (solo) clip for the clearest read.
demo_record = next((r for r in records if r['recording'].endswith('_solo')), records[0])

for t, c in [('standard', 0), ('drop_d', 0), ('standard', 2)]:
    demo_tab(demo_record, tuning=t, capo=c)
    print()

## 7. Drop D — before/after the position anchor

The motivating case. With no anchor and the prior inert off-standard, a geometry-only decode can
ride high up one string on a stepwise run (correct, low-movement, but not where a guitarist would
sit). Anchoring toward a target region pulls the register back down **only when it's worth it** —
the term is a preference, so a genuinely good high run survives. Same notes, anchor vs no anchor.

In [9]:
demo_record = next((r for r in records if r['recording'].endswith('_solo')), records[0])

set_preferred_position(None)                          # default: no anchor
demo_tab(demo_record, tuning='drop_d')                # geometry-only register

print()
set_preferred_position('fret_9')                      # anchor the hand around fret 9
demo_tab(demo_record, tuning='drop_d')                # tail should settle toward the target region

set_preferred_position(None)                          # restore

# Calibration: start weight 0.30 / window 2 and sweep 0.2-0.6. Raise it until the off-standard
# tail settles into the target region; stop the moment clean stepwise runs start getting shredded
# into string-hops to chase smaller fret numbers. That breakpoint is the sweet spot.

Preferred position: OFF (learned prior governs register)
00_BN1-129-Eb_solo   |   Tuning: drop_d
assigner=assign_viterbi_playability   notes=48
e|------------------------------------------------------------------------------8-----------------------------------------------------------------------------|
B|------------------------------------------------------------6--------11-------------------------------------------------------------------------------------|
G|---3-----------------------------------7-----7--------7-----------7--10-10----12----------15-17-------------------------------------------------------------|
D|6-----2-----------7--8--7--------7-----------------8--------------------------13----15----------20-20-19-------------17----17----15-17-15-13-12-13-15-13----|
A|6--8--7-----8-----7--------8--------------------------------------------------------------------------------------------------------------------------------|
E|8-----------------------------------------------------

## Notes, caveats & what's next

**Capo** is exactly correct — pure transposition plus a fret window; every cost is position-relative,
so behavior matches standard. The transpose-equivalence test is the evidence.

**Alternate tunings** are correct for candidate generation and any geometry-based assigner. Two things
degrade *gracefully*: the **learned position prior goes inert** off-standard (keys are standard-tuning
triples → flat default), so geometry and the position anchor govern register; and in the **CAGED line**
specifically, `VOICING_SHAPES` and `LOW_E_PC` assume standard intervals, so chords on a retuned string
(e.g. anything on the dropped low string in Drop D/C) would get a wrong voicing bonus — gate the voicing
bonus behind `ACTIVE_TUNING_NAME == 'standard'`, or demo alternate tunings on melodic passages.

**Prior hardening (one-line, in the base).** If you ever rebuild the prior after switching tunings,
`build_position_prior` would validate standard-tuning ground truth against the wrong fretboard. Pin its
two checks to `STANDARD_TUNING` instead of `OPEN_STRING_MIDI` (the validity check and the `len(...)`).
Not needed for normal top-to-bottom runs, where the prior is built in standard during `%run`.

**Tab export.** `demo_tab` returns the render-ready dict, so
`open(path,'w').write(render_ascii_tab(parsed))` with `tab_capo_header()` as the first line is the file
the site serves. That header line is also where to fix the export-target bug — point ASCII export at
`caged_voiced`, not `combined_all_tuned`.

**What turns the position anchor into a reportable result** (the others ship as demos; this ships as a
number). On VAL with ground truth: per clip, set the target to the expert's median true hand-center,
then measure `exact_tab_f1` and mean hand-center deviation *with* the anchor vs *without* — an oracle
upper bound. Sweep the weight on VAL for where F1 holds/improves while the ~69% directional string-bias
shrinks. Slots into `eval_pipeline.ipynb` next to the existing `exact_tab_f1` harness.

**Callback.** The chatbot's "can I play this higher?" is `set_preferred_position('high')` + re-decode.
The substance is here; the conversational wrapper is later and thin.